### Librerias

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import csv
import shutil
import zipfile
from google.colab.patches import cv2_imshow
from google.colab import drive

In [ ]:
def fix_extensions(directory):
    count = 0
    for filename in os.listdir(directory):
        if filename.endswith(".png"):
            base = os.path.splitext(filename)[0]
            os.rename(os.path.join(directory, filename), os.path.join(directory, base + ".jpg"))
            count += 1
    print(f"Se cambiaron {count} archivos de .png a .jpg en: {directory}")

# Aplicar a las carpetas de entrenamiento y validación
fix_extensions('/content/dataset/train/pha')
fix_extensions('/content/dataset/val/pha')

In [ ]:
!python /content/a_hnet_article.py
!python /content/hnet_skipy.py

In [ ]:
# @title Modificar el archivo "hnet_skipy.py"
%%writefile /content/hnet_skipy.py

from __future__ import print_function

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, MaxPooling2D, UpSampling2D, Dropout, Conv2D, Concatenate, Activation, Add, Conv2DTranspose
#from keras.layers.normalization import BatchNormalization
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.regularizers import l2


def DCB(x, concat_axis, nb_filter,dilatation, weight_decay=1E-4, name=None):
    x = Conv2D(nb_filter, (7, 7), dilation_rate=dilatation,kernel_initializer="he_uniform",padding="same",
               kernel_regularizer=l2(weight_decay),name=name+'conv')(x)
    x = Activation('relu')(x)
    x = BatchNormalization(axis=concat_axis,gamma_regularizer=l2(weight_decay),beta_regularizer=l2(weight_decay))(x)


    return x

def convBlock(x, concat_axis,nfilters=None,dropout_rate=None,weight_decay=1E-4,name=None):
    x = Conv2D(6, (5, 5),kernel_initializer="he_uniform",padding="same",
               kernel_regularizer=l2(weight_decay),name=name+'conv')(x)
    x = Activation('relu')(x)
    x = BatchNormalization(axis=concat_axis,gamma_regularizer=l2(weight_decay),beta_regularizer=l2(weight_decay))(x)


    if dropout_rate:
        x = Dropout(dropout_rate)(x)
    return x

def ResconvBlock(x, concat_axis,nfilters=None,dropout_rate=None,weight_decay=1E-4,name=None):
    x_in = Conv2D(6,(1,1),padding="same")(x)
    x = Conv2D(6, (5, 5),kernel_initializer="he_uniform",padding="same",kernel_regularizer=l2(weight_decay),name=name+'resconv')(x)
    x = Activation('relu')(x)
    x = BatchNormalization(axis=concat_axis,gamma_regularizer=l2(weight_decay),beta_regularizer=l2(weight_decay))(x)
    if dropout_rate:
        x = Dropout(dropout_rate)(x)
    x = Add()([x_in,x])
    return x



def denseblock(x, concat_axis, nb_layers, growth_rate,
               dropout_rate=None, weight_decay=1E-4, name=None):
    list_feat = [x]
    for i in range(nb_layers):
        x = convBlock(x, concat_axis, growth_rate,
                         dropout_rate, weight_decay,name=name+"{0}".format(i))
        list_feat.append(x)
        x = Concatenate(axis=concat_axis)(list_feat)

    return x

def get_model(input_channels=3):
    inputs = Input((256,256,input_channels))    # <--Se cambió esta linea

    print("input shape:",inputs.shape)
    dcb_t1_1 = DCB(x=inputs,concat_axis=3,nb_filter=3,dilatation=64,name="A1")
    print("dcb_t1_1 shape:",dcb_t1_1.shape)
    dcb_t1_2 = DCB(x=inputs,concat_axis=3,nb_filter=3,dilatation=16,name="B1")
    print("dcb_t1_2 shape:",dcb_t1_2.shape)
    dcb_t1_3 = DCB(x=inputs,concat_axis=3,nb_filter=3,dilatation=4,name="C1")
    print("dcb_t1_3 shape:",dcb_t1_3.shape)
    dcb_t1_4 = DCB(x=inputs,concat_axis=3,nb_filter=3,dilatation=2,name="D1")
    print("dcb_t1_4 shape:",dcb_t1_4.shape)
    dcb_t1_5 = DCB(x=inputs,concat_axis=3,nb_filter=3,dilatation=8,name="E1")
    print("dcb_t1_5 shape:",dcb_t1_5.shape)
    dcb_t1_6 = DCB(x=inputs,concat_axis=3,nb_filter=3,dilatation=32,name="F1")
    print("dcb_t1_6 shape:",dcb_t1_6.shape)

    dcb_t2_1 = DCB(x=dcb_t1_3,concat_axis=3,nb_filter=3,dilatation=3,name="G1")
    print("dcb_t2_1 shape:",dcb_t2_1.shape)
    dcb_t2_2 = DCB(x=dcb_t1_3,concat_axis=3,nb_filter=3,dilatation=12,name="J1")
    print("dcb_t2_2 shape:",dcb_t2_2.shape)
    dcb_t2_3 = DCB(x=dcb_t1_3,concat_axis=3,nb_filter=3,dilatation=64,name="A2")#equals to dcb_t1_1 dilatation
    print("dcb_t2_3 shape:",dcb_t2_3.shape)
    dcb_t2_4 = DCB(x=dcb_t1_3,concat_axis=3,nb_filter=3,dilatation=32,name="F2")#equals to dcb_t1_6 dilatation
    print("dcb_t2_4 shape:",dcb_t2_4.shape)

    dcb_t2_5 = DCB(x=dcb_t1_4,concat_axis=3,nb_filter=3,dilatation=6,name="I1")
    print("dcb_t2_5 shape:",dcb_t2_5.shape)
    dcb_t2_6 = DCB(x=dcb_t1_4,concat_axis=3,nb_filter=3,dilatation=9,name="H1")
    print("dcb_t2_6 shape:",dcb_t2_6.shape)
    dcb_t2_7 = DCB(x=dcb_t1_4,concat_axis=3,nb_filter=3,dilatation=32,name="F3")#equals to dcb_t1_6 dilatation
    print("dcb_t2_7 shape:",dcb_t2_7.shape)
    dcb_t2_8 = DCB(x=dcb_t1_3,concat_axis=3,nb_filter=3,dilatation=61,name="A3")#equals to dcb_t1_1 dilatation
    print("dcb_t2_8 shape:",dcb_t2_8.shape)

    featMap3 = Add()([dcb_t1_2,dcb_t2_1,dcb_t2_6])
    print("featMap3 shape:",featMap3.shape)
    featMap4 = Add()([dcb_t1_5,dcb_t2_5,dcb_t2_2])
    print("featMap4 shape:",featMap4.shape)


    dcb_t2_9 = DCB(x=featMap3,concat_axis=3,nb_filter=3,dilatation=64,name="A4")#equals to dcb_t1_1 dilatation
    print("dcb_t2_9 shape:",dcb_t2_9.shape)
    dcb_t2_11 = DCB(x=featMap3,concat_axis=3,nb_filter=3,dilatation=32,name="F4")#equals to dcb_t1_6 dilatation
    print("dcb_t2_11 shape:",dcb_t2_11.shape)

    dcb_t2_10 = DCB(x=featMap4,concat_axis=3,nb_filter=3,dilatation=32, name = "F5")#equals to dcb_t1_6 dilatation
    print("dcb_t2_10 shape:",dcb_t2_10.shape)
    dcb_t2_12 = DCB(x=featMap4,concat_axis=3,nb_filter=3,dilatation=64,name="A5")#equals to dcb_t1_1 dilatation
    print("dcb_t2_12 shape:", dcb_t2_12.shape)

    featMap5 = Add()([dcb_t1_1,dcb_t2_9,dcb_t2_3,dcb_t2_12,dcb_t2_8])
    print("featMap5 shape:",featMap5.shape)
    featMap6 = Add()([dcb_t1_6,dcb_t2_4,dcb_t2_7,dcb_t2_11,dcb_t2_10])
    print("featMap6 shape:",featMap6.shape)


    cb1= ResconvBlock(x = dcb_t1_3, concat_axis=3,name="cb1")
    print("cb1 shape:", cb1.shape)
    cb2 = ResconvBlock(x = dcb_t1_4, concat_axis=3,name="cb2")
    print("cb2 shape:", cb2.shape)
    cb3 = ResconvBlock(x = featMap3, concat_axis=3,name="cb3")
    print("cb3 shape:", cb3.shape)
    cb4 = ResconvBlock(x=featMap4,concat_axis=3,name="cb4")
    print("cb4 shape:", cb4.shape)
    cb5 = ResconvBlock(x = featMap5, concat_axis=3,name="cb5")
    print("cb5 shape:", cb5.shape)
    cb6 = ResconvBlock(x = featMap6, concat_axis=3,name="cb6")
    print("cb6 shape:", cb6.shape)

    merge1 = Concatenate(axis=3)([dcb_t1_1, cb1])
    merge2 = Concatenate(axis=3)([dcb_t1_2, cb2])
    merge3 = Concatenate(axis=3)([dcb_t1_3, cb3])
    merge4 = Concatenate(axis=3)([dcb_t1_4, cb4])
    merge5 = Concatenate(axis=3)([dcb_t1_5, cb5])
    merge6 = Concatenate(axis=3)([dcb_t1_6, cb6])


    #outCb = Add()([cb1,cb2,cb3,cb4,cb5,cb6])
    outCb = Add()([merge1,merge2,merge3,merge4,merge5,merge6])
    print("outCb shape:",outCb.shape)
    outCbdense = denseblock(x=outCb, concat_axis=3, nb_layers=3, growth_rate=16, dropout_rate=0.5,name="dense")
    print("outCbdense shape:",outCbdense)
    outh = Conv2D(10,(3,3), activation='relu',padding="same",name="outbefore")(outCbdense)
    print("outh shape:",outh.shape)
    out = Conv2D(1,(5,5),activation="sigmoid",padding="same",name="outconv")(outh)
    print("out shape:",out.shape)
    model = Model(inputs=inputs, outputs=out)
    return model

In [ ]:
import tensorflow as tf

from hnet_skipy import get_model   # o a_hnet_article

model = get_model()
model.summary()

In [ ]:
def LR(epoch):
    if epoch < 40:
        return 0.0001
    elif epoch < 80:
        return 0.00001
    else:
        return 0.000001

BS = 16
Epocas = 15
path = "/content/drive/MyDrive/Estancia_Profesional/checkpoint/"

In [ ]:
from tensorflow.keras.saving import register_keras_serializable
@register_keras_serializable()
def dice_bce_loss(y_true, y_pred, smooth=1e-6):
    alpha = 0.01
    gamma = 0.99
    # ── BCE ───────────────────────────────────────────────────
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)

    # ── Dice (DSC) ────────────────────────────────────────────
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    dsc = (2.0 * intersection + smooth) / (
            tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

    return alpha * bce - gamma * dsc

@register_keras_serializable()
def dice_metric(y_true, y_pred, smooth=1e-6):
    y_true_f     = tf.keras.backend.flatten(y_true)
    y_pred_f     = tf.keras.backend.flatten(y_pred)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2.0 * intersection + smooth) / (
            tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss= dice_bce_loss,
    metrics=[
        dice_metric,
        tf.keras.metrics.IoU(num_classes=2, target_class_ids=[0,1])
        ]
)

In [ ]:
for layer in model.layers[:]:
    layer.trainable = True

model.compile(  # recompilar tras cambiar trainable
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),  # LR más bajo
    loss=  dice_bce_loss,
    metrics=[
        dice_metric,
        tf.keras.metrics.IoU(num_classes=2, target_class_ids=[0,1])
    ]
)

In [ ]:
import glob
# ── 1. Recolectar rutas y ordenarlas  ───
image_paths = sorted(glob.glob("/content/dataset/train/fgr/*.jpg"))
mask_paths  = sorted(glob.glob("/content/dataset/train/pha/*.jpg"))
image_val = sorted(glob.glob("/content/dataset/val/fgr/*.jpg"))
mask_val  = sorted(glob.glob("/content/dataset/val/pha/*.jpg"))

In [ ]:
def load_data(image_path, mask_path):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.cast(img, tf.float32) / 255.0
    img = tf.image.resize(img, [256, 256])

    # normalización ImageNet
    mean = tf.constant([0.485, 0.456, 0.406])
    std  = tf.constant([0.229, 0.224, 0.225])
    img  = (img - mean) / std

    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=1)
    mask = tf.cast(mask, tf.float32) / 255.0
    mask = tf.image.resize(mask, [256, 256])
    mask = tf.round(mask)

    return img, mask

#entrenamiento
dataset = tf.data.Dataset.from_tensor_slices((image_paths, mask_paths))
dataset = dataset.map(load_data).batch(BS).prefetch(tf.data.AUTOTUNE)
#validación
val_dataset = tf.data.Dataset.from_tensor_slices((image_val, mask_val))
val_dataset = val_dataset.map(load_data).batch(BS).prefetch(tf.data.AUTOTUNE)

In [ ]:
import time

class TiempoRestante(tf.keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs=None):
        self._epoch_start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        elapsed     = time.time() - self._epoch_start
        epochs_left = self.params['epochs'] - (epoch + 1)
        eta         = elapsed * epochs_left

        # formatear a hh:mm:ss
        def fmt(s):
            h, r = divmod(int(s), 3600)
            m, s = divmod(r, 60)
            return f"{h:02d}:{m:02d}:{s:02d}"

        print(f"  ⏱  Época {epoch+1}/{self.params['epochs']} "
              f"— duración: {fmt(elapsed)} "
              f"— tiempo restante estimado: {fmt(eta)}")

In [ ]:
# ── Guardar checkpoint completo (pesos + optimizer + época) ────
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    filepath=f"{path}checkpoint_epoch_train.keras",
    save_weights_only=False,   # guarda todo, no solo pesos
    save_best_only=False,      # guardar en cada época
    verbose=1
)

# guardar solo guardar el mejor
checkpoint_best = tf.keras.callbacks.ModelCheckpoint(
    filepath=f"{path}best_model.keras",
    save_weights_only=False,
    save_best_only=True,
    monitor="val_loss",
    verbose=1
)

In [ ]:
callbacks = [
    tf.keras.callbacks.LearningRateScheduler(LR, verbose=1),
    checkpoint_cb,
    checkpoint_best,
    TiempoRestante()
]

history = model.fit(
    dataset,
    validation_data=val_dataset,
    epochs=Epocas,
    callbacks=callbacks)

guardar_history(history, "/content/drive/MyDrive/Estancia_Profesional/")

In [ ]:
import json

def guardar_history(history, path, nombre="history.json"):
    historia_path = os.path.join(path, nombre)

    # cargar history previo si existe
    if os.path.exists(historia_path):
        with open(historia_path, "r") as f:
            historia_previa = json.load(f)
        # concatenar
        for key in history.history:
            if key in historia_previa:
                historia_previa[key].extend(history.history[key])
            else:
                historia_previa[key] = history.history[key]
        historia_combinada = historia_previa
    else:
        historia_combinada = history.history

    with open(historia_path, "w") as f:
        json.dump(historia_combinada, f)
    print(f"History guardado en {historia_path}")
    return historia_combinada

def cargar_history(path, nombre="history.json"):
    historia_path = os.path.join(path, nombre)
    if os.path.exists(historia_path):
        with open(historia_path, "r") as f:
            return json.load(f)
    print("No se encontró history previo")
    return {}

In [ ]:
# ── Recargar y continuar ───────────────────────────────────────
import tensorflow as tf

# cargar el modelo completo (arquitectura + pesos + estado del optimizer)
model = tf.keras.models.load_model(
    f"{path}checkpoint_epoch_train.keras",
    custom_objects={
        "loss": dice_bce_loss,
        "dice_metric":   dice_metric
    }
)

# verificar en qué época quedó
print("LR actual:", model.optimizer.learning_rate.numpy())
historia_acumulada = cargar_history("/content/drive/MyDrive/Estancia_Profesional/")

In [ ]:
callbacks = [
    tf.keras.callbacks.LearningRateScheduler(LR, verbose=1),
    checkpoint_cb,
    checkpoint_best,
    TiempoRestante()
]
# continuar  hasta terminar
history2 = model.fit(
    dataset,
    validation_data=val_dataset,
    epochs=100,               # época FINAL, no cantidad adicional # 35  60  85
    initial_epoch=85,        # <-- continúa desde aquí            # 15  35  60
    callbacks=callbacks
)
guardar_history(history2, "/content/drive/MyDrive/Estancia_Profesional/")